# Condition C: case-only Qwen validation pilot

Run this notebook first. It reuses the fitted PHM classifier and three retrieved historical training cases per query. It does not process literature. Charan is excluded.

The local export contains ten fixed validation cases. Start with the first three, one per class, to check the workflow. Query labels are used only for selection and later scoring, never sent to Qwen. This is a development pilot, not the final C/D experiment.

Select **Runtime > Change runtime type > T4 GPU**. Start a fresh runtime if the literature model is still loaded. Run the cells in order. The model is **Qwen/Qwen3-4B-Instruct-2507**, pinned in `experiment_config.json`; no Hugging Face token is required for this public model.

In [ ]:
from pathlib import Path
import os
import sys
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not (PROJECT_ROOT / 'Files' / 'phm_diagnosis_qwen.py').is_file():
    raise FileNotFoundError('The new C files have not synced, or PROJECT_ROOT differs from your Drive folder.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'Files'))
from phm_run_reporting import RunReport
report = RunReport(PROJECT_ROOT, 'condition_C_validation_pilot')
report.install_notebook_hooks(get_ipython())
config = report.configure(PROJECT_ROOT / 'Files' / 'experiment_config.json')
BUNDLE = PROJECT_ROOT / 'Datasets/PHM15-diagnosis/validation_pilot_v1'
MAX_CASES = 3
RUN_C = True
RETRY_ERRORS = False
print('Condition: C only')
print('Project:', PROJECT_ROOT)
print('Run report:', report.folder)

## Install the pinned inference packages

This keeps Colab's existing CUDA-enabled PyTorch. It does not install or load the saved numerical model. If Colab asks for a session restart after installation, restart and rerun from the setup cell.

In [ ]:
report.run_command([sys.executable, '-m', 'pip', 'install', '-r',
                    str(PROJECT_ROOT / 'Files' / 'requirements-qwen-colab.txt')],
                   name='install_Qwen_dependencies')
report.refresh_packages()
report.save()

## Check the portable evidence and software

Every prompt includes all 31 selected descriptors, fixed regression scores and the same three historical cases. Arrays use the saved feature order to reduce repeated text. Display values use six significant digits; full-precision evidence is retained in the export. Nothing is silently truncated.

In [ ]:
from phm_diagnosis import prepare_requests
with report.step('check_portable_C_inputs'):
    contract, records, requests = prepare_requests(BUNDLE, config, MAX_CASES)
    report.check('validation_only', contract['scope'] == 'validation_pilot')
    report.check('three_training_cases_each', all(len(r['historical_cases']) == 3 for r in records))
    print('Selected queries:', len(records), 'of', len(contract['case_ids']))
    print('Native features:', contract['native_features'])
    print('Query IDs:', [r['case_id'] for r in records])
report.run_command([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files',
                    '-p', 'test_phm_diagnosis.py', '-v'], name='diagnostic_software_checks')

## Run the three-case C pilot

The first run downloads the model weights. Each response is saved immediately. Repeating the same inputs and settings reuses completed responses. Leave `RETRY_ERRORS=False` for the initial run; failures remain visible for review. Any deliberate retry archives the previous attempt.

Automatic checks validate JSON, cited IDs and quoted values. They do not establish whether the full explanation is supported. We will review the actual outputs before expanding the pilot.

In [ ]:
from phm_diagnosis_qwen import run_condition_C
if RUN_C:
    result = run_condition_C(PROJECT_ROOT, max_cases=MAX_CASES,
                             retry_errors=RETRY_ERRORS, reporter=report)
    print(result['pilot_metrics'])
else:
    report.note('CPU setup only: RUN_C=False; no diagnostic generation requested.')
    print('Setup checked. Rerun from the setup cell with RUN_C=True when ready.')

## Save the review package

After Drive sync finishes, tell Codex: **C pilot finished—check the latest report.** The review package includes exact prompts, raw responses, validation/scoring results and runtime information. Do not rerun preprocessing or the literature notebook.

After reviewing these three cases, we can expand to the ten fixed validation cases, then build D using permitted, approved literature. D must reuse C's query evidence and historical cases. Neither final test scoring nor an independent LLM judge is run here.

In [ ]:
folder = report.finish()
print('Status:', report.data['status'])
print('Saved report:', folder)
print('Latest pointer:', PROJECT_ROOT / 'Outputs' / 'LATEST_RUN.json')
print('Response cache:', PROJECT_ROOT / 'Outputs' / 'diagnosis' / 'C' / 'responses')